In [ ]:
import pandas as pd
import sys
from pathlib import Path

# Get project root (one level up from notebooks/)
PROJECT_ROOT = Path("..").resolve()

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

# Now import
from env_config import AUDIO_ROOT

In [44]:
df = pd.read_csv("../data/metadata/raga_20_dataset.csv")
df.head()

,track_id,artist,raga,tradition,relative_part
0,Abhishek_Raghuram.Nidhi_Chala_Sukhama,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...
1,Abhishek_Raghuram.Isha_Paahimam,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...
2,Sanjay_Subrahmanyan.Vanajakshi_-_Varnam,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...
3,Sanjay_Subrahmanyan.Paarengum,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...
4,Sanjay_Subrahmanyan.Ambarame_Tannire,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...


In [45]:
df["audio_path"] = df["relative_part"].apply(lambda x: AUDIO_ROOT / x)
df.head()

,track_id,artist,raga,tradition,relative_part,audio_path
0,Abhishek_Raghuram.Nidhi_Chala_Sukhama,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...
1,Abhishek_Raghuram.Isha_Paahimam,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...
2,Sanjay_Subrahmanyan.Vanajakshi_-_Varnam,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...
3,Sanjay_Subrahmanyan.Paarengum,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...
4,Sanjay_Subrahmanyan.Ambarame_Tannire,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...


In [47]:
df["raga"].value_counts()


raga
Kalyāṇi            12
Kāmavardani        12
Ṣanmukhapriya      12
Dhanyāsi           12
Tōḍi               12
Sahānā             12
Bilahari           12
Bhairavi           12
Mōhanaṁ            12
Karaharapriya      12
Rītigauḷa          12
Kāṁbhōji           12
Harikāmbhōji       12
Nāṭakurinji        12
Madhyamāvati       12
Śankarābharaṇaṁ    12
Ānandabhairavi     12
Māyāmāḷavagauḷa    12
Kāpi               12
Bēgaḍa             11
Name: count, dtype: int64

In [48]:
df.isnull().sum()

track_id         0
artist           0
raga             0
tradition        0
relative_part    0
audio_path       0
dtype: int64

In [49]:
df['track_id'].duplicated().sum()

0

In [50]:
import os

def fix_long_path(p):
    if os.name == "nt":
        p = os.path.normpath(p)  # convert / → \ properly
        
        if len(p) >= 260 and not p.startswith("\\\\?\\"):
            return "\\\\?\\" + p
    return p

df['audio_path'] = df['audio_path'].apply(fix_long_path)

In [42]:
import os

# Check if all audio paths exist
missing_paths = []
bad = []
for path in df['audio_path']:
    if not os.path.exists(path):
        missing_paths.append(path)
    
long_paths = [p for p in df['audio_path'] if len(p) > 260]
print(len(long_paths))

print(f"Total audio files: {len(df)}")
print(f"Missing audio files: {len(missing_paths)}")

if missing_paths:
    print("\nFirst few missing paths:")
    for path in missing_paths[:5]:
        print(f"  - {path}")
else:
    print("\nAll audio paths exist!")

73
Total audio files: 235
Missing audio files: 0

All audio paths exist!


In [51]:
df = df[df['audio_path'].apply(os.path.exists)].reset_index(drop=True)
print(len(df))

235


We will split the data based on artists so that there is no leakage of information.  
We assign an entire artist to train/ test / val set. 


In [52]:
# raga mapping 
artist_ragas = (
    df.groupby("artist")["raga"]
    .unique()
    .to_dict()
)
artist_ragas

{'Abhishek_Raghuram': array(['Kalyāṇi', 'Madhyamāvati', 'Nāṭakurinji', 'Kāṁbhōji', 'Bilahari',
        'Ṣanmukhapriya'], dtype=object),
 'Akkarai_Sisters': array(['Rītigauḷa', 'Ṣanmukhapriya'], dtype=object),
 'Amritha_Murali': array(['Kāmavardani', 'Māyāmāḷavagauḷa', 'Ānandabhairavi', 'Bēgaḍa',
        'Nāṭakurinji', 'Kāṁbhōji', 'Karaharapriya', 'Bhairavi', 'Sahānā',
        'Tōḍi', 'Kāpi'], dtype=object),
 'Amrutha_Venkatesh': array(['Kalyāṇi', 'Kāmavardani', 'Māyāmāḷavagauḷa', 'Śankarābharaṇaṁ',
        'Rītigauḷa', 'Mōhanaṁ', 'Bilahari', 'Sahānā'], dtype=object),
 'Aruna_Sairam': array(['Māyāmāḷavagauḷa', 'Ānandabhairavi', 'Madhyamāvati',
        'Harikāmbhōji', 'Rītigauḷa', 'Karaharapriya', 'Sahānā', 'Dhanyāsi'],
       dtype=object),
 'Bombay_Jayashri': array(['Bēgaḍa', 'Bhairavi', 'Tōḍi'], dtype=object),
 'Chaitra_Sairam': array(['Ānandabhairavi', 'Mōhanaṁ'], dtype=object),
 'D__K__Jayaraman': array(['Bhairavi', 'Bilahari'], dtype=object),
 'Dr_Pantula_Rama': array(['Nāṭakurinji

In [53]:
# List artists for each raga 
from collections import defaultdict

raga_artists = defaultdict(set)

for artist, ragas in artist_ragas.items():
    for r in ragas:
        raga_artists[r].add(artist)
raga_artists

defaultdict(set,
            {'Kalyāṇi': {'Abhishek_Raghuram',
              'Amrutha_Venkatesh',
              'Gayathri_Venkataraghavan',
              'Modhumudi_Sudhakar',
              'Nisha_P_Rajagopal',
              'Sanjay_Subrahmanyan',
              'Sikkil_Gurucharan',
              'Vijay_Siva'},
             'Madhyamāvati': {'Abhishek_Raghuram',
              'Aruna_Sairam',
              'Gayathri_Venkataraghavan',
              'Lalgudi_GJR_Krishnan',
              'Modhumudi_Sudhakar',
              'O__S__Thyagarajan',
              'Ramnad_Krishnan',
              'S__Somasundaram',
              'S__Sowmya',
              'T__M__Krishna'},
             'Nāṭakurinji': {'Abhishek_Raghuram',
              'Amritha_Murali',
              'Dr_Pantula_Rama',
              'Gayathri_Girish',
              'Nisha_P_Rajagopal',
              'Nithyasree_Mahadevan',
              'Prasanna_Venkataraman',
              'Sanjay_Subrahmanyan',
              'Sikkil_Gurucharan',

In [64]:
#Split such that train has all ragas and test and val have most 

import random
random.seed(0)

train_artists = set()
test_artists = set()
val_artists = set()

for raga, artists in raga_artists.items():
    artists= list(artists)
    random.shuffle(artists)

    if artists[0] not in train_artists | val_artists | test_artists:
        train_artists.add(artists[0])
    if len(artists)>= 2:
        if artists[1] not in train_artists | val_artists | test_artists:
            val_artists.add(artists[1])
    if len(artists)>= 3:
        if artists[2] not in train_artists | val_artists | test_artists:
            test_artists.add(artists[2])


In [65]:
assigned= train_artists | val_artists | test_artists
remaining_artists = set(artist_ragas.keys()) - assigned
artist_track_count= df.groupby("artist").size().to_dict()

def split_track_load(artists):
    return sum(artist_track_count[a] for a in artists)

# Assign remaining artists one by one
for artist in remaining_artists:
    split_loads = [
        (split_track_load(train_artists), train_artists),
        (split_track_load(val_artists),   val_artists),
        (split_track_load(test_artists),  test_artists),
    ]
# print(f"Remaining artists not assigned to any split: {remaining_artists}")
# len(remaining_artists)

    target_split = min(split_loads, key=lambda x: x[0])[1]
    target_split.add(artist)

In [66]:
df['split'] = None

In [67]:
df.loc[df["artist"].isin(train_artists), "split"] = "train"
df.loc[df["artist"].isin(val_artists),   "split"] = "val"
df.loc[df["artist"].isin(test_artists),  "split"] = "test"

In [68]:
assert (
    df.groupby("artist")["split"].nunique().max() == 1
)

In [69]:
# ensure no track is left unassigned
df["split"].isna().sum()

print(df["split"].value_counts())


split
train    80
test     78
val      77
Name: count, dtype: int64


In [70]:
train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]
test_df = df[df["split"] == "test"]

In [71]:
df.head()

,track_id,artist,raga,tradition,relative_part,split
0,Abhishek_Raghuram.Nidhi_Chala_Sukhama,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,val
1,Abhishek_Raghuram.Isha_Paahimam,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,val
2,Sanjay_Subrahmanyan.Vanajakshi_-_Varnam,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,test
3,Sanjay_Subrahmanyan.Paarengum,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,test
4,Sanjay_Subrahmanyan.Ambarame_Tannire,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,test


In [72]:
df.drop(columns=["audio_path"], inplace=True)

KeyError: "['audio_path'] not found in axis"

In [73]:
df.to_csv("../data/metadata/raga_20_dataset_frozen.csv")